# CelebA Generator — Simple Word Embeddings + Diffusion
Train a small **TensorFlow/Keras conditional diffusion model from scratch**, then enter:
- `smiling male`
- `a female with glasses`
- `smiling female with blond hair and no glasses`

**Start a fresh runtime and run all six code cells.** The old unconditional weights are incompatible. This version uses **20,000 faces**, **30,000 training steps**, and **32 × 32 RGB** by default. These settings are a stronger baseline, not a verified image-quality or runtime guarantee. Full training is not benchmarked.

The code keeps the full learning story in one notebook: **attributes → trainable word embeddings → noise predictor → iterative generation**.

## 1. Load faces AND their 40 attribute labels
The original image-only mirror did not supply the labels we need. We now stream the **training split of `flwrlabs/celeba`**, whose schema includes all 40 original binary attributes. We use the first 20,000 training rows and cache their resized images and labels together.

`Male=False` is the source dataset's complement used for the word `female`; it is not a separate annotation. These original appearance labels are imperfect and some are subjective. See [VOCABULARY.md](VOCABULARY.md) for all 40 attributes and prompt spellings.

Keep SIZE=32 while learning recognizable face shapes. SIZE=64 is an optional later experiment and requires preparing new data and training a fresh model. More pixels alone do not solve undertraining.

In [ ]:
import importlib.util, subprocess, sys, re
if importlib.util.find_spec('datasets') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets>=3,<5'])
from pathlib import Path
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from datasets import load_dataset, Image as DatasetImage
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)
NUM_IMAGES = 20000  # Four times the old dataset size.
SIZE = 32          # Learn face structure first; optional 64 needs a fresh model.
ATTRIBUTES = """5_o_Clock_Shadow Arched_Eyebrows Attractive Bags_Under_Eyes Bald Bangs
Big_Lips Big_Nose Black_Hair Blond_Hair Blurry Brown_Hair Bushy_Eyebrows Chubby
Double_Chin Eyeglasses Goatee Gray_Hair Heavy_Makeup High_Cheekbones Male
Mouth_Slightly_Open Mustache Narrow_Eyes No_Beard Oval_Face Pale_Skin Pointy_Nose
Receding_Hairline Rosy_Cheeks Sideburns Smiling Straight_Hair Wavy_Hair
Wearing_Earrings Wearing_Hat Wearing_Lipstick Wearing_Necklace Wearing_Necktie Young""".split()
cache = Path(f'celeba_attributes_v3_{NUM_IMAGES}_{SIZE}.npz')  # Ignore old image-only caches.
if cache.exists():
    with np.load(cache, allow_pickle=False) as saved:
        images, attributes = saved['images'], saved['attributes']
        assert list(saved['attribute_names']) == ATTRIBUTES, 'Wrong attribute order in cache.'
else:
    stream = load_dataset('flwrlabs/celeba', 'img_align+identity+attr', split='train',
        streaming=True, revision='2d738f56e0e7f925ea36ae7c808ea925264aacec')
    stream = stream.cast_column('image', DatasetImage(decode=False))
    pictures, labels = [], []
    for row in stream.take(NUM_IMAGES):
        with Image.open(BytesIO(row['image']['bytes'])) as original:
            face = ImageOps.fit(original.convert('RGB'), (SIZE, SIZE),
                method=Image.Resampling.LANCZOS, bleed=0.1)
            pictures.append(np.asarray(face))
        labels.append([int(row[name] > 0) for name in ATTRIBUTES])  # Present=1, absent=0.
        if len(pictures) % 1000 == 0:
            print('Faces and labels prepared:', len(pictures))
    images, attributes = np.stack(pictures).astype('uint8'), np.asarray(labels, dtype='uint8')
    np.savez_compressed(cache, images=images, attributes=attributes, attribute_names=ATTRIBUTES)
image_bank, attribute_bank = tf.constant(images), tf.constant(attributes, dtype=tf.int32)
print('Images:', images.shape, '| Attributes:', attributes.shape)
for name in ['Smiling', 'Male', 'Eyeglasses']:
    print(name, ':', int(attributes[:, ATTRIBUTES.index(name)].sum()), 'positive examples')

def show_images(pictures, title):
    pictures = np.clip((np.asarray(pictures) + 1) / 2, 0, 1)
    fig, axes = plt.subplots(1, len(pictures), figsize=(2 * len(pictures), 2.5), squeeze=False)
    for ax, picture in zip(axes[0], pictures):
        ax.imshow(picture, interpolation='nearest'); ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_images(images[:6].astype('float32') / 127.5 - 1, 'Actual training faces')

## 2. Prompt words → attribute tokens → learned embeddings
CelebA has **40 binary attributes**, not unrestricted text captions. Positive and negative states give **80 attribute-state tokens**, plus **token 0 for unspecified**. Aliases such as `glasses`/`eyeglasses` share a token.

Examples: `smiling male` sets Smiling=True and Male=True. `a female with glasses` sets Male=False and Eyeglasses=True. All other attributes remain **unspecified**, not False. `no`, `not`, and `without` negate the following concept.

Each token will select **32 trainable numbers** from an Embedding layer. This controlled parser does not understand arbitrary English. Unknown words and contradictory conditions raise an error; the interpreted conditions are printed before generation.

In [ ]:
# Every official attribute name also works with spaces instead of underscores.
aliases = {name.lower().replace('_', ' '): (i, 1) for i, name in enumerate(ATTRIBUTES)}
# Aliases change the wording, not the number of annotated concepts.
for word, name, value in [
    ('female', 'Male', 0), ('woman', 'Male', 0), ('man', 'Male', 1),
    ('glasses', 'Eyeglasses', 1), ('smile', 'Smiling', 1),
    ('blonde hair', 'Blond_Hair', 1), ('grey hair', 'Gray_Hair', 1),
    ('beard', 'No_Beard', 0), ('clean shaven', 'No_Beard', 1),
    ('hat', 'Wearing_Hat', 1), ('lipstick', 'Wearing_Lipstick', 1),
    ('earrings', 'Wearing_Earrings', 1), ('necklace', 'Wearing_Necklace', 1),
    ('necktie', 'Wearing_Necktie', 1)]:
    aliases[word] = (ATTRIBUTES.index(name), value)

def encode_prompt(prompt):
    words = re.sub(r'[^a-z0-9 ]', ' ', prompt.lower().replace('_', ' ').replace('-', ' ')).split()
    tokens, interpreted, i = np.zeros(40, dtype='int32'), {}, 0
    while i < len(words):
        if words[i] in {'a', 'an', 'the', 'with', 'and', 'face', 'person'}:
            i += 1; continue
        negative = words[i] in {'no', 'not', 'without'}
        i += int(negative)
        # Longest match handles multiword concepts such as "black hair".
        match = next((phrase for phrase in sorted(aliases, key=len, reverse=True)
                      if words[i:i + len(phrase.split())] == phrase.split()), None)
        if match is None:
            raise ValueError('Unsupported or incomplete prompt near: ' + ' '.join(words[i:]))
        attribute, value = aliases[match]
        value = 1 - value if negative else value
        token = 1 + 2 * attribute + value  # 1..80; zero always means unspecified.
        if tokens[attribute] not in (0, token):
            raise ValueError('Conflicting conditions for ' + ATTRIBUTES[attribute])
        tokens[attribute] = token
        interpreted[ATTRIBUTES[attribute]] = bool(value)
        i += len(match.split())
    if not interpreted:
        raise ValueError('Enter at least one supported attribute, e.g. smiling male.')
    return tokens, interpreted

print(encode_prompt('a female with glasses')[1])

## 3. Add known noise
\[
x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon.
\]
$x_0$ is a clean face scaled to −1 to 1; $\epsilon$ is known Gaussian noise. We use 1,000 cosine-scheduled noise levels. The network is shared across all levels, rather than training 1,000 networks.

In [ ]:
T = 1000
curve = np.cos((np.linspace(0, 1, T + 1) + 0.008) / 1.008 * np.pi / 2) ** 2
curve /= curve[0]
beta = np.clip(1 - curve[1:] / curve[:-1], 0, 0.999)
alpha_bar = tf.constant(np.cumprod(1 - beta), dtype=tf.float32)

def add_noise(clean, t, noise):
    a = tf.reshape(tf.gather(alpha_bar, t), [-1, 1, 1, 1])
    return tf.sqrt(a) * clean + tf.sqrt(1 - a) * noise

clean = tf.cast(image_bank[:1], tf.float32) / 127.5 - 1
noise = tf.random.normal(tf.shape(clean))
frames = [clean] + [add_noise(clean, [t], noise) for t in [0, 200, 400, 600, 800, 999]]
show_images(tf.concat(frames, axis=0), 'Clean face → increasing noise')

## 4. A stronger conditional U-Net
The model receives **noisy pixels, time, and attribute tokens**. It predicts the added noise. Compared with the previous model, residual connections and group normalization improve the learning path, and sine/cosine time features distinguish noise levels more clearly.

Word embeddings are learned jointly with the network. An EMA model stores a moving average of the training weights and is used for generation. This adds a few lines but is useful for steadier samples. Both models start untrained.

In [ ]:
@keras.utils.register_keras_serializable()
def sinusoidal_time(t):
    frequencies = tf.exp(tf.linspace(tf.math.log(1.0), tf.math.log(1000.0), 32))
    angles = t * frequencies[None, :] * (2 * np.pi)
    return tf.concat([tf.sin(angles), tf.cos(angles)], axis=-1)

noisy_input = keras.Input((SIZE, SIZE, 3), name='noisy_face')
time_input = keras.Input((1,), name='noise_level')
word_input = keras.Input((40,), dtype='int32', name='attribute_tokens')
words = layers.Flatten()(layers.Embedding(81, 32, name='word_embedding')(word_input))
time = layers.Lambda(sinusoidal_time, output_shape=(64,))(time_input)
condition = layers.Dense(128, activation='swish')(layers.Concatenate()([words, time]))
condition = layers.Dense(128, activation='swish')(condition)

def block(x, channels):
    residual = layers.Conv2D(channels, 1)(x) if x.shape[-1] != channels else x
    h = layers.Activation('swish')(layers.GroupNormalization(groups=8)(x))
    h = layers.Conv2D(channels, 3, padding='same')(h)
    hint = layers.Reshape((1, 1, channels))(layers.Dense(channels)(condition))
    h = layers.Activation('swish')(layers.Add()([h, hint]))
    h = layers.Conv2D(channels, 3, padding='same')(h)
    return layers.Add()([residual, h])

a = block(block(layers.Conv2D(48, 3, padding='same')(noisy_input), 48), 48)
b = block(block(layers.AveragePooling2D(2)(a), 96), 96)
h = block(block(layers.AveragePooling2D(2)(b), 144), 144)
h = block(block(layers.Concatenate()([layers.UpSampling2D()(h), b]), 96), 96)
h = block(block(layers.Concatenate()([layers.UpSampling2D()(h), a]), 48), 48)
h = layers.Activation('swish')(layers.GroupNormalization(groups=8)(h))
output = layers.Conv2D(3, 3, padding='same', kernel_initializer='zeros')(h)
model = keras.Model([noisy_input, time_input, word_input], output)
ema_model = keras.models.clone_model(model)
ema_model.set_weights(model.get_weights())
print('Trainable parameters:', model.count_params())

## 5. Train with partial descriptions
**30,000 additional steps**, batch 64. The model learns the same noise-MSE objective as before.

For each face we randomly expose only some annotated attributes, so short prompts do not imply that all other features are absent. Ten percent of examples hide every attribute; this teaches an unconditional prediction for classifier-free guidance.

EMA weights are saved every 5,000 steps and when training stops. This saves weights only, not a complete optimizer checkpoint. Rerunning this cell continues the current in-memory model and optimizer. Recreating the model resets it. Do not reuse the old unconditional model weights.

In [ ]:
STEPS = 30000  # Six times the old optimization budget; no runtime guarantee.
BATCH_SIZE = 64
if 'optimizer_model' not in globals() or optimizer_model is not model:
    optimizer = keras.optimizers.Adam(learning_rate=0.0002, clipnorm=1.0)
    optimizer_model = model

@tf.function
def train_step():
    ids = tf.random.uniform([BATCH_SIZE], 0, len(images), dtype=tf.int32)
    clean = tf.cast(tf.gather(image_bank, ids), tf.float32) / 127.5 - 1
    clean = tf.image.random_flip_left_right(clean)  # Simple data augmentation.
    labels = tf.gather(attribute_bank, ids)
    tokens = 1 + 2 * tf.range(40)[None, :] + labels
    keep_rate = tf.random.uniform([BATCH_SIZE, 1], 0.05, 0.30)
    known = tf.random.uniform([BATCH_SIZE, 40]) < keep_rate
    known &= tf.random.uniform([BATCH_SIZE, 1]) >= 0.10  # Some fully unconditional rows.
    tokens = tf.where(known, tokens, 0)  # Unknown is different from a negative attribute.
    t = tf.random.uniform([BATCH_SIZE], 0, T, dtype=tf.int32)
    noise = tf.random.normal(tf.shape(clean))
    noisy = add_noise(clean, t, noise)
    time = tf.cast(t[:, None], tf.float32) / (T - 1)
    with tf.GradientTape() as tape:
        prediction = model([noisy, time, tokens], training=True)
        loss = tf.reduce_mean(tf.square(noise - prediction))
    optimizer.apply_gradients(zip(tape.gradient(loss, model.trainable_variables), model.trainable_variables))
    for average, weight in zip(ema_model.weights, model.weights):
        average.assign(0.995 * average + 0.005 * weight)
    return loss

losses = []
try:
    for step in range(STEPS):
        losses.append(float(train_step()))
        if (step + 1) % 100 == 0:
            print(f'Step {step + 1}/{STEPS} | noise MSE {np.mean(losses[-100:]):.4f}')
        if (step + 1) % 5000 == 0:
            ema_model.save_weights('celeba_conditioned_ema.weights.h5')
finally:
    ema_model.save_weights('celeba_conditioned_ema.weights.h5')
plt.plot(losses); plt.xlabel('Training step'); plt.ylabel('Noise MSE'); plt.show()

## 6. Enter a prompt and generate
Use `smiling male` or `a female with glasses`. Keep the seed fixed when comparing prompts or training progress. The network outputs are not photo filters or retrieved training examples.

Guidance 1 uses the conditional prediction directly; modest guidance emphasizes the specified attributes. Excessive guidance can distort faces. Here $\hat\epsilon=\hat\epsilon_u+w(\hat\epsilon_c-\hat\epsilon_u)$.

We use **100 DDIM steps**. Leading timestep spacing starts near, rather than exactly at, the almost-zero-signal endpoint; this avoids dividing the initial clean-image estimate by an extremely tiny signal multiplier. Pure-noise initialization is an approximation at this selected near-terminal level. This addresses a numerical weakness in the earlier sampler, but it cannot replace learning a good noise predictor.

In [ ]:
PROMPT = 'smiling male'  #@param {type:"string"}
SEED = 42  #@param {type:"integer"}
GUIDANCE = 1.5  #@param {type:"number"}
N = 6
encoded, interpreted = encode_prompt(PROMPT)
print('Interpreted conditions:', interpreted)
tokens = tf.constant(np.repeat(encoded[None], N, axis=0))
x = tf.random.stateless_normal([N, SIZE, SIZE, 3], seed=[SEED, 0])
times = np.arange(0, T, T // 100)[::-1]  # 990, 980, ..., 0; skip the singular endpoint.
frames = [x[:1]]

@tf.function
def guided_noise(x, time, tokens):
    # One batch contains both the unconditional and conditional predictions.
    both = ema_model([tf.concat([x, x], 0), tf.concat([time, time], 0),
                      tf.concat([tf.zeros_like(tokens), tokens], 0)], training=False)
    unconditional, conditional = tf.split(both, 2)
    return unconditional + GUIDANCE * (conditional - unconditional)

for i, t in enumerate(times):
    time = tf.fill([N, 1], tf.cast(t, tf.float32) / (T - 1))
    noise = guided_noise(x, time, tokens)
    a = alpha_bar[t]
    clean = tf.clip_by_value((x - tf.sqrt(1 - a) * noise) / tf.sqrt(a), -1, 1)
    noise = (x - tf.sqrt(a) * clean) / tf.sqrt(1 - a)
    previous_a = alpha_bar[times[i + 1]] if i + 1 < len(times) else tf.constant(1.0)
    x = tf.sqrt(previous_a) * clean + tf.sqrt(1 - previous_a) * noise
    if (i + 1) % 20 == 0:
        frames.append(x[:1])
show_images(x, PROMPT + ' — EMA model output')
show_images(tf.concat(frames, axis=0), 'Actual noisy states: noise → face')

### If faces still look unclear
- Check the **actual training examples** first. At 32×32, fine detail will always be limited, but recognizable face structure is still the goal.
- Run more training, then compare fixed seeds and prompts. A falling noise MSE alone does not prove realism or prompt adherence.
- Glasses and other rare labels have fewer examples. Inspect the printed counts; increasing data is useful but does not guarantee that rare attributes will be followed.
- If memory is limited, lower BATCH_SIZE to 32. To try 64×64 later, change SIZE, prepare that cache and train a fresh model.
- Download `celeba_conditioned_ema.weights.h5` before a runtime reset. After rebuilding the same architecture, `ema_model.load_weights(...)` restores inference weights. To initialize a fresh training run from them, load them into both model and ema_model; optimizer state is not restored.

This update improves the training setup and implements attribute conditioning. Full 30,000-step generation quality and training duration have **not** been benchmarked. CelebA source terms still apply; images are not redistributed in the repository.

References: [CelebA](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html), [Keras DDIM example](https://keras.io/examples/generative/ddim/), [DDIM](https://arxiv.org/abs/2010.02502), [classifier-free guidance](https://arxiv.org/abs/2207.12598).